# Notebook 11 — Mini-Project: Wright-Fisher vs. Hardy-Weinberg

**Module:** 06 — Genetics and Evolution  
**Notebook:** 11 of 12  
**Portfolio artifact:** `portfolio/module06_wright_fisher_vs_hardy_weinberg.png`  
**Prerequisites:** Module 05 NB06–NB08, Module 06 NB04–NB05  
**Time estimate:** 120 minutes

> **This is a portfolio figure.** Implement everything from memory before checking
> previous notebooks. The goal is fluency, not novelty.

---
## Project Brief

Produce a single 4-panel publication-ready figure that demonstrates the contrast
between HWE (the null model — no drift, no selection, infinite population) and the
Wright-Fisher model (finite population with optional selection). The figure should
be self-contained: a stranger who knows population genetics but hasn't seen this
code should be able to understand all four panels from their titles and axis labels alone.

### Panel specification

| Panel | Content | Key parameter varied |
|-------|---------|---------------------|
| A | WF allele frequency trajectories, N={50, 500, 5000} | Population size N |
| B | HWE expected vs. observed genotype frequencies after drift | Generations of drift |
| C | Fixation probability: Kimura formula vs. WF simulation | Selection coefficient s |
| D | Time to fixation distribution, neutral (s=0) | Population size N |

### Implementation rules

1. All functions implemented from scratch — do not copy from previous notebooks
2. All random seeds set explicitly for reproducibility
3. All panels labelled A–D with `ax.text(-0.1, 1.05, 'A', transform=ax.transAxes, ...)`
4. Figure saved to `portfolio/module06_wright_fisher_vs_hardy_weinberg.png` at 150 dpi
5. Reflection paragraph written before saving

---
## Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

os.makedirs('../../portfolio', exist_ok=True)

In [ ]:
# ── IMPLEMENT FROM MEMORY ─────────────────────────────────────────────────────
# All functions below must be implemented without copying from previous notebooks.
# Write them first, then run the figure cell.

def wright_fisher(N: int, p0: float, n_gen: int, n_trials: int,
                  s: float = 0.0, seed: int = 42) -> np.ndarray:
    """
    Wright-Fisher simulation with optional selection.
    Returns: ndarray shape (n_trials, n_gen+1)
    """
    rng = np.random.default_rng(seed)
    trajectories = np.zeros((n_trials, n_gen + 1))
    trajectories[:, 0] = p0

    for trial in range(n_trials):
        p = p0
        for gen in range(n_gen):
            if s != 0:
                q = 1 - p
                p = p * (1 + s) / (p * (1 + s) + q)
            p = rng.binomial(2 * N, p) / (2 * N)
            trajectories[trial, gen + 1] = p

    return trajectories


def hwe_expected(p: float) -> dict:
    """HWE genotype frequencies given allele frequency p."""
    q = 1 - p
    return {'AA': p**2, 'Aa': 2*p*q, 'aa': q**2}


def kimura_fixation_prob(p0: float, s: float, Ne: int) -> float:
    """Kimura (1962) fixation probability."""
    if abs(s) < 1e-10:
        return p0
    num = 1 - np.exp(-2 * Ne * s * p0)
    den = 1 - np.exp(-2 * Ne * s)
    return num / den


def time_to_fixation(trajectories: np.ndarray) -> np.ndarray:
    """
    For each trajectory, return the generation when allele first reached
    fixation (p=1) or loss (p=0). Returns NaN if neither occurred.
    """
    n_trials, n_gen = trajectories.shape
    fix_times = np.full(n_trials, np.nan)
    for i in range(n_trials):
        fixed_at = np.where((trajectories[i] >= 1.0) | (trajectories[i] <= 0.0))[0]
        if len(fixed_at) > 0:
            fix_times[i] = fixed_at[0]
    return fix_times

In [ ]:
# ── Pre-compute data for all panels ─────────────────────────────────────────

N_sizes = [50, 500, 5000]
n_gen = 200
p0 = 0.4
n_trials_traj = 30  # trajectories per N

# Panel A: trajectories
traj_data = {N: wright_fisher(N, p0, n_gen, n_trials_traj, seed=42) for N in N_sizes}

# Panel B: HWE vs. post-drift genotype frequencies
N_drift = 50
gen_snapshots = [0, 5, 20, 100]
traj_B = wright_fisher(N_drift, p0, 100, 1000, seed=7)
hwe_obs_data = {}
for g in gen_snapshots:
    p_vals = traj_B[:, g]
    p_mean = p_vals.mean()
    hwe_obs_data[g] = {
        'p_mean': p_mean,
        'AA_obs': (p_vals**2).mean(),
        'Aa_obs': (2*p_vals*(1-p_vals)).mean(),
        'aa_obs': ((1-p_vals)**2).mean(),
        'hwe': hwe_expected(p_mean),
    }

# Panel C: Kimura vs. simulation fixation probability
Ne_C = 200
p0_C = 0.05
s_range = np.linspace(-0.05, 0.10, 20)
kimura_pfix = [kimura_fixation_prob(p0_C, s, Ne_C) for s in s_range]

sim_pfix = []
for s in s_range:
    traj_sim = wright_fisher(Ne_C, p0_C, 20*Ne_C, 800, s=s, seed=13)
    fixed_frac = (traj_sim[:, -1] >= 1.0).mean()
    sim_pfix.append(fixed_frac)

# Panel D: time to fixation distribution
fix_time_data = {}
for N in [50, 100, 200]:
    traj_fix = wright_fisher(N, 1/(2*N), 40*N, 2000, s=0.0, seed=99)
    fix_gen = time_to_fixation(traj_fix)
    # Keep only those that actually fixed (p=1)
    fixed_mask = traj_fix[:, -1] >= 1.0
    if fixed_mask.sum() > 10:
        fix_time_data[N] = fix_gen[fixed_mask]

print("Data computed. Creating figure...")

In [ ]:
# ── Figure ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.33)

panel_labels = ['A', 'B', 'C', 'D']
axes = [fig.add_subplot(gs[i//2, i%2]) for i in range(4)]

colors_N = {'50': 'tomato', '500': 'steelblue', '5000': 'seagreen'}

# ── Panel A: WF trajectories ──────────────────────────────────────────────
ax = axes[0]
t = np.arange(n_gen + 1)
for N in N_sizes:
    color = colors_N[str(N)]
    for trial in range(n_trials_traj):
        alpha = 0.15
        ax.plot(t, traj_data[N][trial], color=color, lw=0.7, alpha=alpha)
    # Label
    ax.plot([], [], color=color, lw=1.5, label=f'N={N}')
ax.axhline(p0, color='black', ls='--', lw=1, label=f'p₀={p0}')
ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency p')
ax.set_title('Genetic drift: frequency trajectories\nsmaller N → more rapid drift')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(-0.02, 1.02)

# ── Panel B: HWE vs. observed genotype freqs ─────────────────────────────
ax = axes[1]
genotypes = ['AA', 'Aa', 'aa']
colors_gt = ['#2196F3', '#4CAF50', '#F44336']
x = np.arange(len(gen_snapshots))
width = 0.12
for gi, (gtype, color) in enumerate(zip(genotypes, colors_gt)):
    obs_vals = [hwe_obs_data[g][f'{gtype}_obs'] for g in gen_snapshots]
    hwe_vals = [hwe_obs_data[g]['hwe'][gtype] for g in gen_snapshots]
    offset = (gi - 1) * width * 2
    ax.bar(x + offset - width/2, obs_vals, width, color=color, alpha=0.75,
           label=f'{gtype} (obs)')
    ax.bar(x + offset + width/2, hwe_vals, width, color=color, alpha=0.35,
           hatch='//', edgecolor=color, label=f'{gtype} (HWE)')

ax.set_xticks(x)
ax.set_xticklabels([f'Gen {g}' for g in gen_snapshots])
ax.set_ylabel('Genotype frequency')
ax.set_title(f'HWE deviation due to drift (N={N_drift})\nSolid=observed, hatched=HWE expected')
ax.legend(fontsize=6, ncol=2, loc='upper right')

# ── Panel C: Kimura vs. simulation ────────────────────────────────────────
ax = axes[2]
ax.plot(s_range, kimura_pfix, color='steelblue', lw=2.5, label='Kimura formula')
ax.scatter(s_range, sim_pfix, color='tomato', s=25, zorder=4, label='WF simulation')
ax.axhline(p0_C, color='gray', ls=':', lw=1, label=f'Neutral limit p₀={p0_C}')
ax.axvline(0, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Selection coefficient s')
ax.set_ylabel('Fixation probability')
ax.set_title(f'Fixation probability: theory vs. simulation\n(Ne={Ne_C}, p₀={p0_C})')
ax.legend(fontsize=8)

# ── Panel D: Time to fixation distribution ────────────────────────────────
ax = axes[3]
color_list = ['tomato', 'steelblue', 'seagreen']
for (N, times), color in zip(fix_time_data.items(), color_list):
    expected_fix = 4 * N  # E[T_fix | fixation] for neutral new mutation ≈ 4Ne
    ax.hist(times / N, bins=30, alpha=0.55, color=color, density=True,
            label=f'N={N} (n={len(times)} fixed)')
    ax.axvline(4, color=color, ls='--', lw=1.5, alpha=0.8)

ax.set_xlabel('Fixation time / N (normalised generations)')
ax.set_ylabel('Density')
ax.set_title('Neutral fixation time distribution\ndashed lines: E[T_fix/N] ≈ 4 (theory)')
ax.legend(fontsize=8)

# Panel labels
for ax, label in zip(axes, panel_labels):
    ax.text(-0.10, 1.05, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

fig.suptitle('Wright-Fisher Model vs. Hardy-Weinberg Equilibrium\n'
             'Module 06 — Genetics and Evolution Portfolio Figure',
             fontsize=12, y=1.01)

plt.savefig('../../portfolio/module06_wright_fisher_vs_hardy_weinberg.png',
            dpi=150, bbox_inches='tight')
print("Figure saved to portfolio/module06_wright_fisher_vs_hardy_weinberg.png")
plt.show()

---
## Reflection

**Date completed:** ____________________

> *Write a paragraph (3–5 sentences) explaining what this figure shows to someone who
> knows basic biology but not population genetics. Which panel was hardest to implement
> correctly? What would you add to the figure if this were for a manuscript?*

---
*Next: `12_module_assessment.ipynb`*